# ControlNet + latent diffusion (img2img)

Part of **FL-YOLO-DADP** (Chen et al., *Improved Federated Learning with Differential Privacy for YOLO Detector on Phase Images*, CDSR 2025). This is one of the four generative baselines in Table 1.

The generator learns to map a 3-channel bounding-box mask (one channel per class: RBC, WBC, PLT) to a phase image. It is trained on real phase images with YOLO labels (`generator_train`), then used to generate a synthetic version of every frame of the three agents' train/valid splits.

**Cells:** *Train* → *Synthesize* (visual check on one frame) → *Generate* (writes `../data/synthetic_raw/controlnet_img2img/agent_<k>_synthesized/`). Afterwards, from the repository root:

```bash
python data_prep/build_synthetic_dataset.py --raw data/synthetic_raw/controlnet_img2img --agents data/agents --out data/synthetic/controlnet_img2img
python baselines/train_yolo.py --data data/synthetic/controlnet_img2img/data.yaml --epochs 20 --name synthetic_controlnet_img2img
```

Paths are relative to this folder (`generative_models/`):

| path | content |
|---|---|
| `../data/generator_train/{train,valid,test}` | real frames + YOLO labels used to train the generator |
| `../data/agents/agent_<k>` | per-agent shards (input of *Generate*) |
| `../data/synthetic_raw/controlnet_img2img` | output of *Generate* |
| `../runs/generative_models/controlnet_img2img` | generator checkpoints |

Starts from Stable Diffusion v1.5 (VAE + U-Net, frozen) and the ControlNet `lllyasviel/control_v11p_sd15_canny`, both downloaded from the Hugging Face Hub. The paper used `runwayml/stable-diffusion-v1-5`, which has since been removed from the Hub; `SD_MODEL_ID` points to the official mirror with the same weights.


In [ ]:
import os
os.makedirs("../runs/generative_models/controlnet_img2img/model/20epochs", exist_ok=True)

# Stable Diffusion v1.5. The paper used "runwayml/stable-diffusion-v1-5" (since removed from the Hub);
# this is the official mirror of the same weights.
SD_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"


### Train

In [ ]:
import os
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from diffusers import UNet2DConditionModel, AutoencoderKL, ControlNetModel, DDPMScheduler
from PIL import Image
import numpy as np
import random

# Set random seed for reproducibility
seed = 99
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ----------------------------------
# 1. Dataset
# ----------------------------------

class BboxImg2ImgDataset(Dataset):
    def __init__(self, images_dir, labels_dir, size=(512, 384)):
        super().__init__()
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.size = size
        self.filepairs = [
            (os.path.join(images_dir, f"{os.path.splitext(txt)[0]}.png"),
             os.path.join(labels_dir, txt))
            for txt in sorted(os.listdir(labels_dir)) if txt.endswith(".txt")
        ]
        self.w, self.h = size
        self.transform = T.Compose([
            T.Resize(size),
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            T.Lambda(lambda x: torch.clamp(x, -1, 1))
        ])

    def create_mask(self, label_path):
        with open(label_path, "r") as f:
            lines = f.readlines()
        mask = np.zeros((self.h, self.w, 3), dtype=np.uint8)
        colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]
        for line in lines:
            try:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id, x_c, y_c, bw, bh = map(float, parts)
                x_c, y_c = int(x_c * self.w), int(y_c * self.h)
                bw, bh = int(bw * self.w), int(bh * self.h)
                x1 = max(0, x_c - bw // 2)
                y1 = max(0, y_c - bh // 2)
                x2 = min(self.w, x_c + bw // 2)
                y2 = min(self.h, y_c + bh // 2)
                if x2 > x1 and y2 > y1:
                    mask[y1:y2, x1:x2] = colors[int(cls_id) % len(colors)]
            except (ValueError, IndexError):
                continue
        return Image.fromarray(mask)

    def __len__(self):
        return len(self.filepairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.filepairs[idx]
        try:
            init_img = Image.open(img_path).convert("RGB")
            mask_img = self.create_mask(lbl_path)
            
            init_tensor = self.transform(init_img)
            control_tensor = self.transform(mask_img)
            
            if torch.isnan(init_tensor).any() or torch.isnan(control_tensor).any():
                raise ValueError("NaN values detected in tensors")
                
            return init_tensor, control_tensor
        except Exception as e:
            print(f"Error loading data: {e}")
            return torch.zeros((3, self.h, self.w)), torch.zeros((3, self.h, self.w))

# ----------------------------------
# 2. Training and Validation Functions
# ----------------------------------

def train_one_epoch(epoch, controlnet, vae, unet, dataloader, optimizer, noise_scheduler, device, max_train_timestep):
    controlnet.train()
    running_loss = 0.0
    valid_batches = 0
    total_batches = len(dataloader)
    
    pbar = tqdm(total=total_batches, desc=f"Epoch [{epoch+1}/20] [Train]")
    
    def encode_img(img_tensor):
        with torch.no_grad():
            img_tensor = img_tensor.to(device, dtype=torch.float32)
            latents = vae.encode(img_tensor).latent_dist.sample() * 0.18215
            return latents

    def process_control(control_img_tensor, latent_height, latent_width):
        control_img_tensor = control_img_tensor.to(device, dtype=torch.float32)
        return F.interpolate(
            control_img_tensor,
            size=(latent_height * 8, latent_width * 8),
            mode="bilinear",
            align_corners=False
        )

    for init_img_tensor, control_img_tensor in dataloader:
        try:
            if torch.isnan(init_img_tensor).any() or torch.isnan(control_img_tensor).any():
                print("Skipping batch due to NaN values in input tensors.")
                continue

            batch_size = init_img_tensor.shape[0]
            latents_0 = encode_img(init_img_tensor)
            t = torch.randint(0, max_train_timestep, (batch_size,), device=device)
            
            # Process inputs
            latents_0 = latents_0.to(dtype=torch.float32)
            noise = torch.randn_like(latents_0)
            latents_t = noise_scheduler.add_noise(latents_0, noise, t)
            
            # Process control condition
            latent_height, latent_width = latents_0.shape[2:]
            control_down = process_control(control_img_tensor, latent_height, latent_width)
            
            # Create empty text embeddings
            context_embeds = torch.zeros((batch_size, 77, 768), device=device, dtype=torch.float32)
            
            # Forward pass
            cnet_out = controlnet(
                sample=latents_t,
                timestep=t,
                encoder_hidden_states=context_embeds,
                controlnet_cond=control_down,
                return_dict=True
            )
            
            control_pred = cnet_out.down_block_res_samples + [cnet_out.mid_block_res_sample]
            
            unet_out = unet(
                sample=latents_t,
                timestep=t,
                encoder_hidden_states=context_embeds,
                down_block_additional_residuals=control_pred[:-1],
                mid_block_additional_residual=control_pred[-1],
                return_dict=True
            )
            
            # Compute loss
            loss = F.mse_loss(unet_out.sample, noise)
            
            if torch.isnan(loss).any():
                print("Skipping batch due to NaN loss.")
                continue
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(controlnet.parameters(), 1.0)
            optimizer.step()
            
            # Update metrics
            running_loss += loss.item()
            valid_batches += 1
            
            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "avg_loss": f"{running_loss / valid_batches:.4f}"})
            
        except RuntimeError as e:
            print(f"Runtime error: {e}")
            continue
            
    pbar.close()
    
    return running_loss / valid_batches if valid_batches > 0 else 0

def validate(controlnet, vae, unet, dataloader, noise_scheduler, device, max_train_timestep):
    controlnet.eval()
    val_loss = 0.0
    val_acc = 0.0
    valid_batches = 0
    
    pbar = tqdm(total=len(dataloader), desc="Validation")
    
    def encode_img(img_tensor):
        with torch.no_grad():
            img_tensor = img_tensor.to(device, dtype=torch.float32)
            latents = vae.encode(img_tensor).latent_dist.sample() * 0.18215
            return latents

    def process_control(control_img_tensor, latent_height, latent_width):
        control_img_tensor = control_img_tensor.to(device, dtype=torch.float32)
        return F.interpolate(
            control_img_tensor,
            size=(latent_height * 8, latent_width * 8),
            mode="bilinear",
            align_corners=False
        )

    with torch.no_grad():
        for init_img_tensor, control_img_tensor in dataloader:
            try:
                if torch.isnan(init_img_tensor).any() or torch.isnan(control_img_tensor).any():
                    print("Skipping batch due to NaN values in input tensors.")
                    continue

                batch_size = init_img_tensor.shape[0]
                latents_0 = encode_img(init_img_tensor)
                t = torch.randint(0, max_train_timestep, (batch_size,), device=device)
                
                # Process inputs
                latents_0 = latents_0.to(dtype=torch.float32)
                noise = torch.randn_like(latents_0)
                latents_t = noise_scheduler.add_noise(latents_0, noise, t)
                
                # Process control condition
                latent_height, latent_width = latents_0.shape[2:]
                control_down = process_control(control_img_tensor, latent_height, latent_width)
                
                # Create empty text embeddings
                context_embeds = torch.zeros((batch_size, 77, 768), device=device, dtype=torch.float32)
                
                # Forward pass
                cnet_out = controlnet(
                    sample=latents_t,
                    timestep=t,
                    encoder_hidden_states=context_embeds,
                    controlnet_cond=control_down,
                    return_dict=True
                )
                
                control_pred = cnet_out.down_block_res_samples + [cnet_out.mid_block_res_sample]
                
                unet_out = unet(
                    sample=latents_t,
                    timestep=t,
                    encoder_hidden_states=context_embeds,
                    down_block_additional_residuals=control_pred[:-1],
                    mid_block_additional_residual=control_pred[-1],
                    return_dict=True
                )
                
                # Compute loss
                loss = F.mse_loss(unet_out.sample, noise)
                
                if torch.isnan(loss).any():
                    print("Skipping batch due to NaN loss.")
                    continue
                
                # Update metrics
                val_loss += loss.item()
                val_acc += 1 - loss.item()  # Placeholder for accuracy
                valid_batches += 1
                
                pbar.update(1)
                pbar.set_postfix({"val_loss": f"{loss.item():.4f}", "val_acc": f"{val_acc / valid_batches:.4f}"})
                
            except RuntimeError as e:
                print(f"Runtime error: {e}")
                continue
            
    pbar.close()
    
    return val_loss / valid_batches if valid_batches > 0 else 0, val_acc / valid_batches if valid_batches > 0 else 0

# ----------------------------------
# 3. Training Loop
# ----------------------------------

def train_controlnet_img2img():
    # Setup
    batch_size = 1
    num_epochs = 20
    learning_rate = 1e-5
    max_train_timestep = 500
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Paths
    train_images_dir = "../data/generator_train/train/images"
    train_labels_dir = "../data/generator_train/train/labels"
    val_images_dir = "../data/generator_train/valid/images"
    val_labels_dir = "../data/generator_train/valid/labels"
    save_dir = "../runs/generative_models/controlnet_img2img/model/20epochs"
    os.makedirs(save_dir, exist_ok=True)
    
    # Dataset and dataloader
    train_dataset = BboxImg2ImgDataset(train_images_dir, train_labels_dir, size=(512, 384))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    
    val_dataset = BboxImg2ImgDataset(val_images_dir, val_labels_dir, size=(512, 384))
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    print("Loading models...")
    vae = AutoencoderKL.from_pretrained(
        SD_MODEL_ID,
        subfolder="vae",
        torch_dtype=torch.float32
    ).to(device)
    vae.eval()
    
    unet = UNet2DConditionModel.from_pretrained(
        SD_MODEL_ID,
        subfolder="unet",
        torch_dtype=torch.float32
    ).to(device)
    unet.eval()
    
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_canny",
        torch_dtype=torch.float32
    ).to(device)
    controlnet.train()
    controlnet.enable_gradient_checkpointing()

    noise_scheduler = DDPMScheduler(
        num_train_timesteps=1000,
        beta_start=0.00085,
        beta_end=0.012,
        beta_schedule="scaled_linear",
        clip_sample=False
    )

    optimizer = torch.optim.AdamW(
        controlnet.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0.01
    )

    # Track best validation accuracy
    best_val_acc = -1.0

    print("Starting training...")
    for epoch in range(num_epochs):
        # Training
        avg_train_loss = train_one_epoch(
            epoch,
            controlnet,
            vae,
            unet,
            train_loader,
            optimizer,
            noise_scheduler,
            device,
            max_train_timestep
        )
        
        # Validation
        avg_val_loss, avg_val_acc = validate(
            controlnet,
            vae,
            unet,
            val_loader,
            noise_scheduler,
            device,
            max_train_timestep
        )
        
        print(f"Epoch [{epoch+1}/20] Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}")
        
        # Save last model
        last_model_path = os.path.join(save_dir, f"controlnet_last.pth")
        controlnet.save_pretrained(last_model_path)
        
        # Save best model
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            best_model_path = os.path.join(save_dir, f"controlnet_best.pth")
            controlnet.save_pretrained(best_model_path)
            print(f"Best model saved with Val Acc: {best_val_acc:.4f}")
    
    print(f"Training complete. Last model saved to: {last_model_path}")

if __name__ == "__main__":
    train_controlnet_img2img()

### Synthesize

In [ ]:
import os
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from diffusers import UNet2DConditionModel, AutoencoderKL, ControlNetModel, DDPMScheduler

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_trained_controlnet(model_path, device):
    controlnet = ControlNetModel.from_pretrained(model_path).to(device)
    controlnet.eval()
    return controlnet

def load_pretrained_models(device):
    vae = AutoencoderKL.from_pretrained(
        SD_MODEL_ID,
        subfolder="vae",
        torch_dtype=torch.float32
    ).to(device)
    vae.eval()
    
    unet = UNet2DConditionModel.from_pretrained(
        SD_MODEL_ID,
        subfolder="unet",
        torch_dtype=torch.float32
    ).to(device)
    unet.eval()
    
    return vae, unet

def preprocess_image(image_path, size=(512, 384)):
    image = Image.open(image_path).convert("RGB")
    image = image.resize(size, Image.BILINEAR)
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        T.Lambda(lambda x: torch.clamp(x, -1, 1))
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)
    return image_tensor

def synthesize_image(controlnet, vae, unet, noise_scheduler, image_tensor, num_inference_steps=50):
    controlnet.eval()
    vae.eval()
    unet.eval()
    
    with torch.no_grad():
        # 1) Encode the input image into latents
        latents = vae.encode(image_tensor).latent_dist.sample() * 0.18215
        
        # 2) We use the same image as the "control" (though typically you'd pass a bounding-box mask).
        control_cond = image_tensor
        
        # 3) Dummy text embeddings (all zeros)
        context_embeds = torch.zeros((1, 77, 768), device=device, dtype=torch.float32)
        
        # 4) Sampling steps (partial, only 'num_inference_steps' from the start)
        for t in noise_scheduler.timesteps[:num_inference_steps]:
            cnet_out = controlnet(
                sample=latents,
                timestep=t,
                encoder_hidden_states=context_embeds,
                controlnet_cond=control_cond,
                return_dict=True
            )
            control_pred = cnet_out.down_block_res_samples + [cnet_out.mid_block_res_sample]
            
            unet_out = unet(
                sample=latents,
                timestep=t,
                encoder_hidden_states=context_embeds,
                down_block_additional_residuals=control_pred[:-1],
                mid_block_additional_residual=control_pred[-1],
                return_dict=True
            )
            # Update latents
            latents = noise_scheduler.step(unet_out.sample, t, latents).prev_sample
        
        # 5) Decode latents -> image
        synthesized_image = vae.decode(latents / 0.18215).sample
        synthesized_image = torch.clamp(synthesized_image, -1, 1)
        synthesized_image = (synthesized_image + 1) / 2.0  # [0,1]
        # shape: [1, 3, H, W]

        # Convert to NumPy for post-processing
        synthesized_image = synthesized_image.squeeze(0).permute(1, 2, 0).cpu().numpy()  # (H, W, 3)
        
        return synthesized_image

def test_synthesis():
    model_path = "../runs/generative_models/controlnet_img2img/model/20epochs/controlnet_best.pth"
    controlnet = load_trained_controlnet(model_path, device)
    vae, unet = load_pretrained_models(device)
    
    noise_scheduler = DDPMScheduler(
        num_train_timesteps=1000,
        beta_start=0.00085,
        beta_end=0.012,
        beta_schedule="scaled_linear",
        clip_sample=False
    )
    
    # 1) Load your test grayscale image
    test_image_path = "../data/generator_train/test/images/test-agg1-2067.png"
    image_tensor = preprocess_image(test_image_path)
    
    # 2) Generate
    synthesized_image = synthesize_image(controlnet, vae, unet, noise_scheduler, image_tensor)
    print(f"synthesized_image shape: {synthesized_image.shape}, range=({synthesized_image.min():.4f},{synthesized_image.max():.4f})")
    
    # 3) Force all channels identical
    #    We'll average across the 3 channels and replicate back:
    gray = synthesized_image.mean(axis=2, keepdims=True)  # shape (H,W,1)
    synthesized_image_gray = np.repeat(gray, 3, axis=2)   # shape (H,W,3)
    
    # 4) Check channel identity
    identical_across_channels = (synthesized_image_gray[:,:,0] == synthesized_image_gray[:,:,1]).all() \
                                and (synthesized_image_gray[:,:,1] == synthesized_image_gray[:,:,2]).all()
    if identical_across_channels:
        print("Now all channels are identical across each pixel.")
    else:
        print("Something went wrong; channels differ.")
    
    # 5) Plot
    plt.subplot(1,2,1)
    plt.title("Raw Output")
    plt.imshow(synthesized_image)  # might have color differences
    plt.axis("off")
    plt.subplot(1,2,2)
    plt.title("Post-Processed to Grayscale")
    plt.imshow(synthesized_image_gray, cmap='gray')  # channels are same
    plt.axis("off")
    plt.show()

if __name__ == "__main__":
    test_synthesis()

### Generate

In [ ]:
import os
import shutil
import torch
import torchvision.transforms as T
import numpy as np
import cv2
from PIL import Image
from diffusers import ControlNetModel, AutoencoderKL, UNet2DConditionModel, DDPMScheduler

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# 1. Load the Trained ControlNet Model
# ---------------------------
def load_trained_controlnet(model_path, device):
    controlnet = ControlNetModel.from_pretrained(model_path).to(device)
    controlnet.eval()  # Set the model to evaluation mode
    return controlnet

# ---------------------------
# 2. Load Pretrained VAE and UNet
# ---------------------------
def load_pretrained_models(device):
    vae = AutoencoderKL.from_pretrained(
        SD_MODEL_ID,
        subfolder="vae",
        torch_dtype=torch.float32
    ).to(device)
    vae.eval()
    
    unet = UNet2DConditionModel.from_pretrained(
        SD_MODEL_ID,
        subfolder="unet",
        torch_dtype=torch.float32
    ).to(device)
    unet.eval()
    
    return vae, unet

# ---------------------------
# 3. Image Preprocessing
# ---------------------------
def preprocess_image(image_path, size=(512, 384)):
    # Load and resize the image
    image = Image.open(image_path).convert("RGB")
    image = image.resize(size, Image.BILINEAR)
    
    # Convert to tensor and normalize
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        T.Lambda(lambda x: torch.clamp(x, -1, 1))
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
    return image_tensor

# ---------------------------
# 4. Latent Image-to-Image Synthesis
# ---------------------------
def synthesize_image(controlnet, vae, unet, noise_scheduler, image_tensor, num_inference_steps=50):
    """
    Synthesize an image using the ControlNet Latent Image-to-Image model.
    """
    controlnet.eval()
    vae.eval()
    unet.eval()
    
    with torch.no_grad():
        # Encode the input image into latents
        latents = vae.encode(image_tensor).latent_dist.sample() * 0.18215
        
        # Prepare the control condition (use the input image as the mask)
        control_cond = image_tensor
        
        # Prepare empty text embeddings
        context_embeds = torch.zeros((1, 77, 768), device=device, dtype=torch.float32)
        
        # Reverse diffusion loop
        for t in noise_scheduler.timesteps[:num_inference_steps]:
            # Predict noise
            cnet_out = controlnet(
                sample=latents,
                timestep=t,
                encoder_hidden_states=context_embeds,
                controlnet_cond=control_cond,
                return_dict=True
            )
            
            control_pred = cnet_out.down_block_res_samples + [cnet_out.mid_block_res_sample]
            
            unet_out = unet(
                sample=latents,
                timestep=t,
                encoder_hidden_states=context_embeds,
                down_block_additional_residuals=control_pred[:-1],
                mid_block_additional_residual=control_pred[-1],
                return_dict=True
            )
            
            # Update latents
            latents = noise_scheduler.step(unet_out.sample, t, latents).prev_sample
        
        # Decode the latents into an image
        synthesized_image = vae.decode(latents / 0.18215).sample
        synthesized_image = torch.clamp(synthesized_image, -1, 1)
        synthesized_image = (synthesized_image + 1) / 2.0  # Scale to [0, 1]
        synthesized_image = synthesized_image.squeeze(0).permute(1, 2, 0).cpu().numpy()  # [H, W, 3]
        gray = np.mean(synthesized_image, axis=2, keepdims=True)    # shape [H,W,1]
        synthesized_image = np.repeat(gray, 3, axis=2)              # shape [H,W,3]
        
        return synthesized_image

# ---------------------------
# 5. Copy Images and Labels
# ---------------------------
def copy_images_and_labels(source_dir, target_dir):
    # Create target directory if it doesn't exist
    os.makedirs(target_dir, exist_ok=True)
    
    # Copy images
    for fname in os.listdir(source_dir):
        if fname.endswith('.png'):
            shutil.copy(os.path.join(source_dir, fname), os.path.join(target_dir, fname))
    
    # Copy labels
    labels_source_dir = source_dir.replace('images', 'labels')
    labels_target_dir = target_dir.replace('images', 'labels')
    os.makedirs(labels_target_dir, exist_ok=True)
    
    for fname in os.listdir(labels_source_dir):
        if fname.endswith('.txt'):
            label_path = os.path.join(labels_source_dir, fname)
            if os.path.exists(label_path):  # Check if the label file exists
                shutil.copy(label_path, os.path.join(labels_target_dir, fname))

# ---------------------------
# 6. Main Function to Generate Dataset
# ---------------------------
def generate_synthesized_dataset():
    # Define base directories
    base_source_dir = "../data/agents"
    base_target_dir = "../data/synthetic_raw/controlnet_img2img"
    
    # List of agents
    agents = ['agent_0', 'agent_1', 'agent_2']
    
    # Load the trained ControlNet model
    model_path = "../runs/generative_models/controlnet_img2img/model/20epochs/controlnet_best.pth"
    controlnet = load_trained_controlnet(model_path, device)
    
    # Load pretrained VAE and UNet
    vae, unet = load_pretrained_models(device)
    
    # Load the noise scheduler
    noise_scheduler = DDPMScheduler(
        num_train_timesteps=1000,
        beta_start=0.00085,
        beta_end=0.012,
        beta_schedule="scaled_linear",
        clip_sample=False
    )
    
    # Process each agent
    for agent in agents:
        # Define source and target directories for training data
        source_images_dir = os.path.join(base_source_dir, agent, 'train', 'images')
        target_images_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'train', 'images')
        
        # Synthesize images for the current agent
        for img_name in os.listdir(source_images_dir):
            if img_name.endswith('.png'):
                img_path = os.path.join(source_images_dir, img_name)
                image_tensor = preprocess_image(img_path)
                synthesized_image = synthesize_image(controlnet, vae, unet, noise_scheduler, image_tensor)
                
                # Save the synthesized image
                target_image_path = os.path.join(target_images_dir, img_name)
                synthesized_image = (synthesized_image * 255).astype(np.uint8)  # Scale to [0, 255]
                cv2.imwrite(target_image_path, cv2.cvtColor(synthesized_image, cv2.COLOR_RGB2BGR))
        
        # Copy images and labels from the other two agents
        for other_agent in agents:
            if other_agent != agent:
                other_source_images_dir = os.path.join(base_source_dir, other_agent, 'train', 'images')
                copy_images_and_labels(other_source_images_dir, target_images_dir)
        
        # Copy labels for all agents
        for other_agent in agents:
            source_labels_dir = os.path.join(base_source_dir, other_agent, 'train', 'labels')
            target_labels_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'train', 'labels')
            copy_images_and_labels(source_labels_dir, target_labels_dir)
    
    # Repeat the process for the validation data
    for agent in agents:
        # Define source and target directories for validation data
        source_images_dir = os.path.join(base_source_dir, agent, 'valid', 'images')
        target_images_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'valid', 'images')
        
        # Synthesize images for the current agent
        for img_name in os.listdir(source_images_dir):
            if img_name.endswith('.png'):
                img_path = os.path.join(source_images_dir, img_name)
                image_tensor = preprocess_image(img_path)
                synthesized_image = synthesize_image(controlnet, vae, unet, noise_scheduler, image_tensor)
                
                # Save the synthesized image
                target_image_path = os.path.join(target_images_dir, img_name)
                synthesized_image = (synthesized_image * 255).astype(np.uint8)  # Scale to [0, 255]
                cv2.imwrite(target_image_path, cv2.cvtColor(synthesized_image, cv2.COLOR_RGB2BGR))
        
        # Copy images and labels from the other two agents
        for other_agent in agents:
            if other_agent != agent:
                other_source_images_dir = os.path.join(base_source_dir, other_agent, 'valid', 'images')
                copy_images_and_labels(other_source_images_dir, target_images_dir)
        
        # Copy labels for all agents
        for other_agent in agents:
            source_labels_dir = os.path.join(base_source_dir, other_agent, 'valid', 'labels')
            target_labels_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'valid', 'labels')
            copy_images_and_labels(source_labels_dir, target_labels_dir)

    print("Dataset generation complete!")

# Run the dataset generation
if __name__ == "__main__":
    generate_synthesized_dataset()
    # 1786m 37.1s1424m 40.5s

In [ ]:
import cv2

# Load the image
test_image_path = "../data/synthetic_raw/controlnet_img2img/agent_0_synthesized/train/images/train_b_cell_11_4_23_cap1_1011.png"
image = cv2.imread(test_image_path)

# Check if the image was loaded successfully
if image is None:
    print("Error: Unable to load image.")
else:
    # Get the number of channels
    num_channels = image.shape[2] if len(image.shape) == 3 else 1
    print(f"The image has {num_channels} channels.")

    # Check if all channels have the same values for each pixel
    if num_channels > 1:
        # Compare all channels
        is_identical = (image[:, :, 0] == image[:, :, 1]).all() and (image[:, :, 1] == image[:, :, 2]).all()
        if is_identical:
            print("All channels have the same values for each pixel.")
        else:
            print("The channels do not have the same values for each pixel.")
    else:
        print("The image is grayscale (only one channel).")